## Sintesi Esecutiva

Un team di operazioni logistiche pianifica una prova sul campo randomizzata che confronta tre strategie di instradamento dei conducenti (base Static, reinstradamento Dynamic, AIOpt ottimizzato con IA) in tre regioni di deposito (Northeast, Midwest, West), con il ritardo medio di consegna (minuti) come risposta. PROC GLMPOWER prende un dataset *esemplificativo* di medie di cella congetturate e risolve per il numero totale di conducenti necessari per rilevare gli effetti di strategia all'80% e al 90% di potenza, quindi mappa come la potenza raggiungibile si deteriora man mano che cresce la variabilità tra percorsi.

# Dimensionamento di una Prova sul Campo di Instradamento dei Conducenti con PROC GLMPOWER

## Sintesi Esecutiva

Un team di operazioni logistiche sta per lanciare una prova sul campo randomizzata che confronta tre strategie di instradamento dei conducenti -- una base **Static**, un sistema di reinstradamento **Dynamic**, e un pianificatore **AIOpt** ottimizzato con IA -- distribuite in tre regioni di deposito (Northeast, Midwest, West). La risposta è il **ritardo medio di consegna in minuti**. Prima di impegnare la capacità della flotta nella prova, il team deve rispondere: *quanti conducenti servono per rilevare in modo affidabile il miglioramento operativo atteso?*

Questo notebook usa **PROC GLMPOWER** per eseguire un'analisi prospettica di potenza e dimensionamento del campione per il modello lineare generale alla base della prova. Partendo da un dataset *esemplificativo* di medie di cella congetturate, esso (1) risolve per l'arruolamento totale necessario per raggiungere l'80% e il 90% di potenza per gli effetti omnibus di strategia e regione, (2) mappa come la potenza raggiungibile si degrada al crescere della variabilità tra percorsi, e (3) produce una curva di potenza a supporto della decisione di arruolamento.

> **Punto chiave:** Con una deviazione standard dell'errore plausibile di 8 minuti, circa **63 conducenti** offrono l'80% di potenza e **83 conducenti** offrono il 90% di potenza per rilevare gli effetti della strategia di instradamento -- ma se la variabilità del ritardo si avvicina a 10 minuti, anche 90 conducenti arruolati non raggiungono il 70% di potenza, evidenziando il valore di percorsi stretti e ben strumentati.

---
## 1. Costruire il Disegno Esemplificativo

Il primo passo codifica la struttura della prova e il ritardo medio *congetturato* dal team per ogni combinazione strategia di instradamento x regione di deposito. Fissiamo un seme casuale e aggiungiamo una variazione trascurabile affinché le medie sembrino realistiche mantenendo la struttura bilanciata 3x3. Il peso `cell_n` pari a 1 in ogni cella indica a GLMPOWER che il disegno è bilanciato.

In [1]:
/* ================================================================
   Genera il dataset esemplificativo dei ritardi medi congetturati.
   Una riga per ogni combinazione strategia di instradamento x regione di deposito.
   ================================================================ */
DATI routing_trial;
   CHIAMARE streaminit(20260531);
   LUNGHEZZA routing_strategy $8 depot_region $9;
   VETTORE strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   VETTORE region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   VETTORE smean[3]     _temporary_ (18.0 14.5 11.0);   /* medie di strategia congetturate */
   VETTORE radj[3]      _temporary_ (1.5  0.0 -1.0);    /* aggiustamenti regionali (min)  */
   FARE i = 1 FINO_A 3;
      FARE j = 1 FINO_A 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         USCITA;
      FINE;
   FINE;
   RIMUOVERE i j;
ESEGUIRE;

PROCEDURA STAMPARE DATI=routing_trial ETICHETTA noobs;
   VARIABILE routing_strategy depot_region mean_delay_min cell_n;
   ETICHETTA routing_strategy="Strategia di instradamento" depot_region="Regione del deposito"
         mean_delay_min="Ritardo medio di consegna (min)" cell_n="Peso della cella";
   TITOLO "Medie di Cella Esemplificative: Ritardo di Consegna Congetturato (minuti)";
ESEGUIRE;

                       Medie di Cella Esemplificative: Ritardo di Consegna Congetturato (minuti)                        

Strategia di instradamento  Regione del deposito  Ritardo medio di consegna (min)  Peso della cella
Static                      Northeast                                19.687507651                 1
Static                      Midwest                                 17.9871067398                 1
Static                      West                                    16.8240210086                 1
Dynamic                     Northeast                               15.9537535365                 1
Dynamic                     Midwest                                 14.4415169858                 1
Dynamic                     West                                    12.8636389493                 1
AIOpt                       Northeast                               12.6143811724                 1
AIOpt                       Midwest                                  10.893885


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Dimensione del Campione per il Disegno Omnibus

Con il disegno pronto, chiediamo a GLMPOWER di **risolvere per la dimensione totale del campione** (`NTOTAL = .`) all'80% e al 90% di potenza. L'istruzione `MODEL` specifica una struttura a due vie con interazione (`routing_strategy*depot_region`), `WEIGHT cell_n` dichiara i pesi di profilo bilanciati, e `STDDEV = 8` è l'errore quadratico medio ipotizzato del ritardo di consegna. `NFRACTIONAL` permette alla procedura di riportare conteggi frazionari esatti prima dell'arrotondamento per eccesso.

Pre-registriamo anche tre confronti `CONTRAST` pianificati -- AI-Opt vs. Static, Dynamic vs. Static, e Qualsiasi reinstradamento vs. Static -- che documentano le ipotesi operativamente rilevanti per cui la prova è costruita.

In [2]:
/* ================================================================
   Risolve per il numero totale di conducenti necessari per raggiungere
   l'80% e il 90% di potenza per gli effetti di strategia di instradamento
   e regione di deposito.
   ================================================================ */
PROCEDURA glmpower DATI=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELLO mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   PESO cell_n;
   ETICHETTA routing_strategy="Strategia di instradamento" depot_region="Regione del deposito"
         mean_delay_min="Ritardo medio di consegna (min)";
   CONTRAST "AI-Opt vs. Static"      routing_strategy -1  0  1;
   CONTRAST "Dynamic vs. Static"     routing_strategy -1  1  0;
   CONTRAST "Qualsiasi reinstradamento vs. Static" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITOLO "Dimensione del Campione per Rilevare gli Effetti della Strategia di Instradamento sul Ritardo";
ESEGUIRE;

                       Medie di Cella Esemplificative: Ritardo di Consegna Congetturato (minuti)                        

The GLMPOWER Procedure


                      Fixed Scenario Elements                      

Item                Value                                          
------------------  -----------------------------------------------
Dependent Variable  Ritardo medio di consegna (min)                
Source              Strategia di instradamento                     
Source              Regione del deposito                           
Source              Strategia di instradamento*Regione del deposito

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Sensibilità della Potenza a Variabilità e Dimensione della Prova

La risposta sulla dimensione del campione dipende dalla deviazione standard dell'errore ipotizzata, che raramente è nota con precisione in anticipo. Qui ribaltiamo la domanda: **fissiamo** diversi totali di arruolamento candidati (`NTOTAL = 45 90 180`) e **risolviamo per la potenza raggiunta** (`POWER = .`) su una griglia di deviazioni standard del ritardo plausibili (6, 8 e 10 minuti). La tabella risultante è una mappa di sensibilità -- mostra quanto è robusto ciascun piano di arruolamento se la variabilità reale dei percorsi risulta più alta del previsto.

In [3]:
/* ================================================================
   Griglia di sensibilità: potenza raggiunta su diverse dimensioni di prova
   candidate e deviazioni standard di ritardo plausibili.
   ================================================================ */
PROCEDURA glmpower DATI=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELLO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   ETICHETTA routing_strategy="Strategia di instradamento" depot_region="Regione del deposito"
         mean_delay_min="Ritardo medio di consegna (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITOLO "Potenza Raggiunta al Variare della Variabilità e degli Scenari di Dimensione della Prova";
ESEGUIRE;

                       Medie di Cella Esemplificative: Ritardo di Consegna Congetturato (minuti)                        

The GLMPOWER Procedure


              Fixed Scenario Elements              

Item                Value                          
------------------  -------------------------------
Dependent Variable  Ritardo medio di consegna (min)
Source              Strategia di instradamento     
Source              Regione del deposito           

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Curva di Potenza per la Decisione di Arruolamento

Infine, tracciamo una **curva di potenza** -- la potenza raggiunta al crescere dell'arruolamento totale da 30 a 270 conducenti in passi di 30 -- per il modello strategia-più-regione alla deviazione standard attesa di 8 minuti. Risolvere `POWER = .` su quella griglia di arruolamento produce la curva come serie tabulata `N Total` vs. `Power`, così possiamo leggere l'arruolamento a cui ciascuno degli obiettivi convenzionali dell'80% e del 90% viene superato.

In [4]:
/* ================================================================
   Curva di potenza: potenza raggiunta vs. totale conducenti arruolati,
   variata da 30 a 270 in passi di 30 alla deviazione standard attesa di 8 min.
   ================================================================ */
PROCEDURA glmpower DATI=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELLO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   ETICHETTA routing_strategy="Strategia di instradamento" depot_region="Regione del deposito"
         mean_delay_min="Ritardo medio di consegna (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITOLO "Curva di Potenza: Potenza Raggiunta vs. Totale Conducenti Arruolati";
ESEGUIRE;

                       Medie di Cella Esemplificative: Ritardo di Consegna Congetturato (minuti)                        

The GLMPOWER Procedure


              Fixed Scenario Elements              

Item                Value                          
------------------  -------------------------------
Dependent Variable  Ritardo medio di consegna (min)
Source              Strategia di instradamento     
Source              Regione del deposito           

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretazione e Raccomandazione

L'analisi fornisce al team di operazioni un piano di arruolamento difendibile:

- **Dimensionamento di base (Sezione 2).** Assumendo una deviazione standard residua di 8 minuti, il modello completo a due vie (strategia, regione e la loro interazione) raggiunge l'**80% di potenza a 63 conducenti** e il **90% di potenza a 83 conducenti**. Arrotondando per l'attrito, un obiettivo di arruolamento vicino a **90 conducenti** supera comodamente la soglia del 90%.

- **La sensibilità conta (Sezione 3).** La potenza è molto sensibile alla variabilità del ritardo. A 90 conducenti, la potenza raggiunta scende da ~99% (SD=6) a ~87% (SD=8) a ~68% (SD=10). Un pilota di 45 conducenti è adeguato solo se la variabilità è bassa (~81% di potenza a SD=6) ed è gravemente sottodimensionato (~37%) se SD raggiunge 10. L'implicazione pratica: investire in percorsi coerenti e ben strumentati per contenere la variabilità è prezioso quanto aggiungere conducenti.

- **La curva di potenza (Sezione 4).** Tracciata per il modello strategia-più-regione (senza termine di interazione, la lente usata per l'analisi di sensibilità), la potenza raggiunta sale da 0.37 a 30 conducenti a 0.69 a 60, 0.87 a 90, e 0.95 a 120, appiattendosi sopra 0.99 a 180. Leggendo la curva rispetto agli obiettivi, l'80% di potenza arriva vicino a 77 conducenti e il 90% vicino a 99 -- leggermente più alto del dimensionamento del modello completo nella Sezione 2 perché eliminare il termine di interazione distribuisce lo stesso effetto su meno gradi di libertà del modello.

**Raccomandazione:** Arruolare circa **90 conducenti** (30 per strategia di instradamento, bilanciati tra le tre regioni di deposito). Questo supera il 90% di potenza per il modello completo sotto la deviazione standard attesa di 8 minuti, mantiene ~87% di potenza anche sulla curva del modello ridotto più conservativa, e mantiene la prova abbastanza piccola da essere eseguita entro un singolo trimestre operativo.

*Nota:* GLMPOWER analizza il *disegno* congetturato, non i risultati osservati -- quindi la credibilità di questi numeri si basa sulle medie e sulla deviazione standard congetturate. I team dovrebbero rivedere il dimensionamento non appena un breve pilota produce una stima empirica della variabilità del ritardo di consegna.